In [12]:
import pandas as pd
from PIL import Image
import io
import torch
from qdrant_client import QdrantClient
from qdrant_client.http import models
from models.scold.model import LVL
from transformers import RobertaTokenizer
from torchvision import transforms
from tqdm.auto import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
df = pd.read_parquet("data/leafnet/leafnet_sampled.parquet")

In [14]:
print(df.head())

                                         file_name  \
0  D:/AI/Dataset/dataset/dataset/079628_PH0119.png   
1  D:/AI/Dataset/dataset/dataset/079551_PH0035.png   
2  D:/AI/Dataset/dataset/dataset/079634_PH0053.png   
3  D:/AI/Dataset/dataset/dataset/079542_PH0307.png   
4  D:/AI/Dataset/dataset/dataset/079536_PH0325.png   

                                               image  \
0  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...   
1  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...   
2  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...   
3  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...   
4  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...   

                                             caption                label  
0  a image of Black Pepper healthy leaves with le...  BlackPepper_Healthy  
1  a image of Black Pepper healthy leaves with le...  BlackPepper_Healthy  
2  a image of Black Pepper healthy leaves with le...  BlackPepper_Healthy  
3  a image of Black Pepper healthy

In [15]:
print(df['caption'].unique())

<StringArray>
[                                                                                                                                                                                         'a image of Black Pepper healthy leaves with leaves appearing normal and healthy',
                                                                                                                'a image of Black Pepper leaves diseased by yellow mottle virus with symptoms of chlorotic, mottled, vein-cleared, and distorted leaves and stunted growth',
                                                                                                                                                    'a image of Cucumber leaves diseased by Downy Mildew with symptoms of yellow to pale green spots and velvety grey fuzz',
 'a image of Maize leaves diseased by Cercospora leaf spot Gray leaf spot with symptoms of small necrotic spots with halos expanding into rectangular gray to brown lesions, typica

In [16]:
model = LVL()
model.load_state_dict(torch.load("models/scold/scold.pt", map_location=device))
model.to(device)
model.eval()

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Andakara\AppData\Local\Temp\ipykernel_63108\3799685660.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_

In [17]:
# Encoding functions
def encode_text(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        # Call text encoder directly instead of going through forward method
        text_emb = model.get_texts_feature(inputs["input_ids"], inputs["attention_mask"])
    return text_emb.cpu().numpy()

def encode_image_from_bytes(image_bytes):
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        # Call image encoder directly instead of going through forward method
        image_emb = model.get_images_features(image_tensor)
    return image_emb.cpu().numpy()

In [18]:
# Initialize Qdrant client
client = QdrantClient(url="http://localhost:6333")
collection_name = "leaf_disease_collection"

In [19]:
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)
    
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "text": models.VectorParams(size=512, distance=models.Distance.COSINE),
        "image": models.VectorParams(size=512, distance=models.Distance.COSINE),
    }
)

True

In [20]:
batch_size = 10  
total_points = []

for i in tqdm(range(0, len(df), batch_size)):
    batch_df = df.iloc[i:i+batch_size]
    batch_points = []
    
    for idx, row in batch_df.iterrows():
        caption_text = str(row['caption'])

        text_vec = encode_text([caption_text])[0]
    
        img_vec = encode_image_from_bytes(row['image']['bytes'])[0]
        
        batch_points.append(models.PointStruct(
            id=idx,
            vector={
                "text": text_vec.tolist(),
                "image": img_vec.tolist(),
            },
            payload={
                "caption": caption_text,
                "label": row['label'],
                "source_id": idx
            }
        ))
        
        total_points.extend(batch_points)
    try:
        client.upsert(
            collection_name=collection_name,
            points=batch_points
        )
    except Exception as e:
        tqdm.write(f"Batch {i//batch_size + 1} error: {e}")
        continue

  0%|          | 0/45 [00:00<?, ?it/s]

In [21]:
def cross_modal_search(query_text=None, image_bytes=None, limit=5):
    if query_text is not None:
        query_vector = encode_text([query_text])[0].tolist()
        search_result = client.query_points(
            collection_name="leaf_disease_collection",
            query=query_vector,
            using="text",
            limit=limit,
            with_payload=True
        )
    elif image_bytes is not None:
        query_vector = encode_image_from_bytes(image_bytes)[0].tolist()
        search_result = client.query_points(
            collection_name="leaf_disease_collection",
            query=query_vector,
            using="image",
            limit=limit,
            with_payload=True
        )
    else:
        raise ValueError("Either query_text or image_bytes must be provided")
    
    return search_result

In [ ]:
text_query = "my apple plant has yellow to bright orange-red spots spots on the leaves"

# Perform search
text_results = cross_modal_search(query_text=text_query, limit=3)

print("Text-to-Image Search Results:")
print("=" * 50)
for i, result in enumerate(text_results.points, 1): 
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Caption: {result.payload['caption']}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Source ID: {result.payload['source_id']}")
    print()

Text-to-Image Search Results:
1. Score: 0.6215
   Caption: a image of Apple leaves diseased by Scab with symptoms of olive-green to dark brown spots on leaves, leading to yellowing
   Label: Apple_Scab
   Source ID: 375

2. Score: 0.6215
   Caption: a image of Apple leaves diseased by Scab with symptoms of olive-green to dark brown spots on leaves, leading to yellowing
   Label: Apple_Scab
   Source ID: 376

3. Score: 0.6215
   Caption: a image of Apple leaves diseased by Scab with symptoms of olive-green to dark brown spots on leaves, leading to yellowing
   Label: Apple_Scab
   Source ID: 377



In [ ]:
s

Image-to-Image Search Results:
1. Score: 0.8397
   Caption: a image of Apple leaves diseased by Cedar-Apple Rust with symptoms of yellow to bright orange-red spots, black dots, and finger-like fungal tubes on the leaf surface.
   Label: Apple_CedarAppleRust
   Source ID: 242

2. Score: 0.7312
   Caption: a image of Apple leaves diseased by Cedar-Apple Rust with symptoms of yellow to bright orange-red spots, black dots, and finger-like fungal tubes on the leaf surface.
   Label: Apple_CedarAppleRust
   Source ID: 240

3. Score: 0.7178
   Caption: a image of Apple leaves diseased by Cedar-Apple Rust with symptoms of yellow to bright orange-red spots, black dots, and finger-like fungal tubes on the leaf surface.
   Label: Apple_CedarAppleRust
   Source ID: 244

